In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/src')

In [ ]:
import numpy as np,pandas as pd
import config as cf, data, predictions

In [ ]:
df = data.build()

rows    4,850,000
series  50,000
dates   2024-03-28 to 2024-07-02
train   4,150,000 2024-03-28 to 2024-06-18
val     350,000 2024-06-19 to 2024-06-25
test    350,000 2024-06-26 to 2024-07-02
Sale amount mean-raw 0.9986
Sale amount mean-recovered 1.2033 (on train data only)

saved: /content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/inputs/daily_demand.parquet


In [ ]:
df = data.load()
print(len(df), 'rows')

4850000 rows


In [ ]:
# install statsforecast
!pip install -q statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.0/501.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 1.1 MB/s eta 0:00:00


In [ ]:
# the two baselines
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, CrostonSBA

BASELINES = {
    'SeasonalNaive': SeasonalNaive(season_length=7),   # last week, repeated
    'CrostonSBA':    CrostonSBA(),                     # intermittent-demand family
}

In [ ]:
# fits every model in the dict, saves one file per model

def run_stats(models,train_df,input_version,split):
  sf=StatsForecast(models=list(models.values()),freq='D',n_jobs=-1)

  fst=sf.forecast(df=train_df[['unique_id','ds','y']],h=cf.HORIZON)

  if 'unique_id' not in fst.columns:
    fst=fst.reset_index()

  for name,model in models.items():
    col=str(model)
    p=fst[['unique_id','ds',col]].rename(columns={col:'prediction'})
    p[['store_id','product_id']]=p['unique_id'].str.split('_',expand=True).astype(int)
    p['dt']=p['ds']
    predictions.save(p[['store_id','product_id','dt','prediction']],name,input_version,split)


In [ ]:
# raw sales, validation week
train_df, predict_df = data.splittbl(df, 'raw', 'val')
run_stats(BASELINES, train_df, 'raw', 'val')

[val] 350,000 rows : SeasonalNaive__raw
[val] 350,000 rows : CrostonSBA__raw


In [ ]:
# recovered demand, validation week
train_df, predict_df = data.splittbl(df, 'recovered', 'val')
run_stats(BASELINES, train_df, 'recovered', 'val')

[val] 350,000 rows : SeasonalNaive__recovered
[val] 350,000 rows : CrostonSBA__recovered


In [ ]:
# raw sales, test week
train_df, predict_df = data.splittbl(df, 'raw', 'test')
run_stats(BASELINES, train_df, 'raw', 'test')

[test] 350,000 rows : SeasonalNaive__raw
[test] 350,000 rows : CrostonSBA__raw


In [ ]:
# recovered demand, test week
train_df, predict_df = data.splittbl(df, 'recovered', 'test')
run_stats(BASELINES, train_df, 'recovered', 'test')

[test] 350,000 rows : SeasonalNaive__recovered
[test] 350,000 rows : CrostonSBA__recovered


In [ ]:
# --- environment record: paste the output back ---
import platform, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

print("OS      :", platform.platform())
print("Python  :", sys.version.split()[0])

try:
    import torch
    print("torch   :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU     :", torch.cuda.get_device_name(0),
              "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
except ImportError:
    print("torch   : not installed")

print("CPU     :", subprocess.run("nproc", capture_output=True, text=True).stdout.strip(), "cores")
print("RAM     :", subprocess.run("free -g | awk 'NR==2{print $2}'", shell=True,
                                  capture_output=True, text=True).stdout.strip(), "GB")

PKGS = ["neuralforecast", "mlforecast", "lightgbm", "statsforecast", "optuna",
        "timesfm", "chronos-forecasting", "pypots", "pandas", "numpy", "pyarrow",
        "scikit-learn", "pytorch-lightning"]
print()
for p in PKGS:
    try:
        print(f"{p:22s} {version(p)}")
    except PackageNotFoundError:
        print(f"{p:22s} -")

OS      : Linux-6.6.122+-x86_64-with-glibc2.35
Python  : 3.12.13
torch   : 2.11.0+cpu | CUDA available: False
CPU     : 2 cores
RAM     : 12 GB

neuralforecast         -
mlforecast             -
lightgbm               4.6.0
statsforecast          2.1.1
optuna                 -
timesfm                -
chronos-forecasting    -
pypots                 -
pandas                 2.2.2
numpy                  2.0.2
pyarrow                18.1.0
scikit-learn           1.6.1
pytorch-lightning      -
